# Knee abnormality detection: using the text reports as a second label sourceThis notebook is the pipeline behind the 0.733 macro AUC run. It is the samemodel as `single-plane-10ep.ipynb` (frozen ConvNeXt-Small + a pooled linearhead); the only thing that changes is **where the training labels come from**.## The problem it solves`train.csv` has 4,407 studies and 12 binary targets, but only **56 studies haveall 12 labels filled in**. The original notebook trains on the 44 of those thatland in the train split and discards the other 4,351 studies, because masked BCEgives an unlabeled row exactly zero gradient. That is the 0.65 ceiling: epochs,augmentation and head size move it by noise only.## The ideaEvery study also has a free-text radiology report, and the report states thefindings the 12 targets ask about. So a study with no image labels is notunlabeled -- it is labeled in a different modality. Reading 1,720 reports andscoring the 12 targets from each one turns 1,720 discarded studies intotraining rows.## Result (5-fold CV over all 56 image-labeled studies, 3 seeds)| training labels | macro AUC (fixed 80 ep) | macro AUC (best-epoch protocol) ||---|---|---|| image labels only | 0.609 +- 0.003 | 0.632 || + 1,720 report labels | **0.733 +- 0.002** | 0.750 |Read the final section before quoting any number from this notebook -- inparticular, why the headline is 0.733 and not the 0.80 that the 12-studyvalidation split reports.

## 0. InputsTwo things must be attached to the session:1. the competition data, as in the original notebook;2. **`weak_labels_by_study.csv`** -- the 1,720 report-derived labels, one row   per `StudyInstanceUID`. Upload it as a Kaggle dataset and point   `WEAK_LABELS_CSV` at it.The labels were produced by reading each report against a fixed rubric(`RUBRIC.md`, written before labeling started) and scoring each target on`{0, 0.25, 0.5, 0.75, 1}`, where the fractions carry radiological hedging:a stated tear is 1, "high-grade partial" is 0.75, "possible small partial" ormucoid degeneration is 0.5, and an incidental mention is 0.25. The targets aretrained with plain BCE, so a fraction acts as a soft target rather than ahard one -- which is the point, because a report that says "suspected tear"should not be taught as a certainty.Re-deriving the labels from scratch needs a model reading 1,720 reports in 12languages; that step is not reproducible inside a Kaggle session, so the CSVis shipped as data. Everything downstream of it runs here.

In [ ]:
import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import pydicom
from PIL import Image
from torchvision import models, transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
WEAK_LABELS_CSV = "/kaggle/input/knee-weak-labels/weak_labels_by_study.csv"
FEATURE_CACHE = "/kaggle/working/feat"

os.makedirs(FEATURE_CACHE, exist_ok=True)

TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

## 1. The label shortage, countedThis is the cell that explains the plateau. If `complete` is 56 out of 4,407,nothing about the model is the bottleneck.

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
train_series = pd.read_csv(os.path.join(DATA_DIR, "train_series.csv"))

complete_mask = train_df[TARGETS].notna().all(axis=1)
any_mask = train_df[TARGETS].notna().any(axis=1)

print("studies in train.csv:          ", len(train_df))
print("studies with ALL 12 labels:    ", int(complete_mask.sum()))
print("studies with >=1 label:        ", int(any_mask.sum()))
print("studies with NO label at all:  ", int((~any_mask).sum()))
print()
print("observed labels per target:")
print(train_df[TARGETS].notna().sum().to_string())

## 2. Series selectionIdentical to the original notebook: sagittal, fluid-sensitive, fat-suppressed,first series per study. This is the single plane the model sees.

In [ ]:
sagittal_fs = train_series[
    (train_series["Anatomical_Plane"].str.lower() == "sagittal")
    & (train_series["Fluid_Sensitive"] == 1)
    & (train_series["Fat_Suppression"] == 1)
]

one_per_study = (
    sagittal_fs
    .sort_values(["StudyInstanceUID", "SeriesInstanceUID"])
    .drop_duplicates("StudyInstanceUID", keep="first")
)

model_df = train_df.merge(
    one_per_study[["StudyInstanceUID", "SeriesInstanceUID"]],
    on="StudyInstanceUID",
    how="inner",
).reset_index(drop=True)

model_df["series_path"] = model_df.apply(
    lambda r: os.path.join(
        DATA_DIR, "train_series",
        r["StudyInstanceUID"], r["SeriesInstanceUID"],
    ),
    axis=1,
)

clean_df = model_df.loc[
    model_df[TARGETS].notna().all(axis=1)
].reset_index(drop=True).copy()

print("studies with a sagittal_fs series:", len(model_df))
print("of those, fully labeled:          ", len(clean_df))

## 3. The report labelsThe merge is on `StudyInstanceUID`, never on row position. Two assertionsmatter and both are cheap:- **no overlap with the 56 image-labeled studies.** The report labels are  training data only; every number in this notebook is measured against image  labels the model has never seen in any form.- **every label is on the rubric grid.** Catches a truncated or shifted file.

In [ ]:
weak = pd.read_csv(WEAK_LABELS_CSV)

assert weak["StudyInstanceUID"].nunique() == len(weak), "duplicate study in weak labels"
assert weak["StudyInstanceUID"].isin(train_df["StudyInstanceUID"]).all(), "unknown study UID"

overlap = set(weak["StudyInstanceUID"]) & set(clean_df["StudyInstanceUID"])
assert not overlap, f"weak labels leak into the evaluation set: {len(overlap)} studies"

grid = {0.0, 0.25, 0.5, 0.75, 1.0}
off_grid = set(np.unique(weak[TARGETS].to_numpy())) - grid
assert not off_grid, f"labels off the rubric grid: {off_grid}"
assert weak[TARGETS].notna().all().all(), "missing values in weak labels"

# keep only the ones whose series we can actually read
weak = weak[weak["StudyInstanceUID"].isin(model_df["StudyInstanceUID"])].reset_index(drop=True)

weak_df = weak.merge(
    model_df[["StudyInstanceUID", "SeriesInstanceUID", "series_path"]],
    on="StudyInstanceUID",
    how="inner",
).reset_index(drop=True)

print("report-labeled studies usable:", len(weak_df))
print("evaluation studies (image labels):", len(clean_df))
print()
print("positive rate per target -- image labels vs report labels:")
cmp = pd.DataFrame({
    "image_56": (clean_df[TARGETS] >= 0.5).mean().round(3),
    "report_1720": (weak_df[TARGETS] >= 0.5).mean().round(3),
})
print(cmp.to_string())

## 4. Frozen features, cached onceThe backbone is frozen, so each slice's 768-d feature is a constant. Computingit once per study and caching the mean-pooled vector turns every laterexperiment from a GPU job into a few seconds of CPU -- which is what made thelabel-count sweep in section 7 affordable at all.Mean pooling over slices is exactly what `ConvNeXtFrozenPool.forward` doeswith a full-length mask, so the cached vector is the same thing the originalmodel would have pooled.**Runtime:** this cell reads DICOMs for ~1,776 studies and is the only slowpart of the notebook -- budget roughly 30-60 minutes on a Kaggle GPU session.It is restartable: already-cached studies are skipped, so a timed-out sessioncan simply be re-run.

In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


def load_dicom_series(series_path):
    # slices in anatomical order, scaled to [0, 1]
    dicoms = []
    for fname in os.listdir(series_path):
        fpath = os.path.join(series_path, fname)
        if not os.path.isfile(fpath):
            continue
        try:
            ds = pydicom.dcmread(fpath)
        except Exception:
            continue
        if not hasattr(ds, "PixelData"):
            continue
        dicoms.append(ds)

    if not dicoms:
        raise RuntimeError(f"no DICOM images found: {series_path}")

    if hasattr(dicoms[0], "ImageOrientationPatient") and hasattr(dicoms[0], "ImagePositionPatient"):
        iop = np.asarray(dicoms[0].ImageOrientationPatient, dtype=np.float32)
        normal = np.cross(iop[:3], iop[3:])
        order = np.argsort([
            float(np.dot(np.asarray(ds.ImagePositionPatient, dtype=np.float32), normal))
            for ds in dicoms
        ])
        dicoms = [dicoms[i] for i in order]
    elif all(hasattr(ds, "InstanceNumber") for ds in dicoms):
        dicoms = sorted(dicoms, key=lambda ds: int(ds.InstanceNumber))

    volume = []
    for ds in dicoms:
        arr = ds.pixel_array.astype(np.float32)
        lo, hi = float(arr.min()), float(arr.max())
        volume.append((arr - lo) / (hi - lo) if hi > lo else np.zeros_like(arr))
    return volume


def build_backbone():
    bb = models.convnext_small(weights=models.ConvNeXt_Small_Weights.DEFAULT)
    bb.classifier = nn.Identity()
    bb.requires_grad_(False)
    return bb.eval().to(DEVICE)


@torch.no_grad()
def cache_features(df, backbone, slice_batch=32):
    done = skipped = 0
    for _, row in df.iterrows():
        sid = row["StudyInstanceUID"]
        out_path = os.path.join(FEATURE_CACHE, f"{sid}.npy")
        if os.path.exists(out_path):
            skipped += 1
            continue
        try:
            volume = load_dicom_series(row["series_path"])
        except Exception as exc:
            print(f"  skip {sid}: {exc}")
            continue
        x = torch.stack([
            image_transform(Image.fromarray(
                (np.clip(s, 0, 1) * 255).astype(np.uint8)
            ).convert("RGB"))
            for s in volume
        ]).to(DEVICE)

        feats = [backbone(x[i:i + slice_batch]).cpu()
                 for i in range(0, len(x), slice_batch)]
        feats = torch.cat(feats).reshape(len(x), -1)
        np.save(out_path, feats.numpy().astype(np.float16))

        done += 1
        if done % 100 == 0:
            print(f"  cached {done} (skipped {skipped})", flush=True)
    print(f"cached {done} new, {skipped} already present")


needed = pd.concat([
    clean_df[["StudyInstanceUID", "series_path"]],
    weak_df[["StudyInstanceUID", "series_path"]],
], ignore_index=True).drop_duplicates("StudyInstanceUID")

print(f"studies to featurize: {len(needed)}")
backbone = build_backbone()
cache_features(needed, backbone)

In [ ]:
def pooled_feature(sid):
    arr = np.load(os.path.join(FEATURE_CACHE, f"{sid}.npy"))
    return arr.astype(np.float32).reshape(-1, 768).mean(0)


have = {f[:-4] for f in os.listdir(FEATURE_CACHE) if f.endswith(".npy")}

clean_use = clean_df[clean_df["StudyInstanceUID"].isin(have)].reset_index(drop=True)
weak_use = weak_df[weak_df["StudyInstanceUID"].isin(have)].reset_index(drop=True)

X_clean = torch.tensor(np.stack([pooled_feature(s) for s in clean_use["StudyInstanceUID"]]))
Y_clean = clean_use[TARGETS].to_numpy(np.float32)

X_weak = torch.tensor(np.stack([pooled_feature(s) for s in weak_use["StudyInstanceUID"]]))
Y_weak = torch.tensor(weak_use[TARGETS].to_numpy(np.float32))

print("evaluation set (image labels):", X_clean.shape)
print("weak training pool:           ", X_weak.shape)

## 5. Head and lossThe head is byte-for-byte the `classifier` of `ConvNeXtFrozenPool`, so anythingtrained here loads back into that class. Two details carried over from theoriginal notebook: `pos_weight` capped at 20, and a per-row weight so thereport labels can be down-weighted relative to the image labels.

In [ ]:
def make_head(seed):
    torch.manual_seed(seed)
    return nn.Sequential(
        nn.LayerNorm(768),
        nn.Linear(768, 256),
        nn.GELU(),
        nn.Dropout(0.3),
        nn.Linear(256, 12),
    )


def compute_pos_weight(Y, max_weight=20.0):
    ws = []
    for i in range(Y.shape[1]):
        v = Y[:, i].numpy()
        pos, neg = int((v == 1).sum()), int((v == 0).sum())
        ws.append(1.0 if (pos == 0 or neg == 0) else min(neg / pos, max_weight))
    return torch.tensor(ws, dtype=torch.float32)


def macro_auc(y_true, y_pred):
    aucs = []
    for i in range(len(TARGETS)):
        ok = np.isfinite(y_true[:, i]) & np.isfinite(y_pred[:, i])
        yt, yp = y_true[ok, i], y_pred[ok, i]
        aucs.append(roc_auc_score(yt, yp) if len(np.unique(yt)) == 2 else np.nan)
    finite = [a for a in aucs if np.isfinite(a)]
    return (float(np.mean(finite)) if finite else np.nan), aucs


EPOCHS, BATCH, LR, WD = 80, 16, 3e-4, 1e-4


def fit(X_tr, Y_tr, W_tr, X_ev, Y_ev, seed):
    # returns (final-epoch preds, best-epoch preds, best fold AUC)
    head = make_head(seed)
    opt = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WD)
    pw = compute_pos_weight(Y_tr)
    gen = torch.Generator().manual_seed(seed + 1000)

    best_auc, best_pred, preds = -1.0, None, None
    for _ in range(EPOCHS):
        head.train()
        perm = torch.randperm(len(X_tr), generator=gen)
        for i in range(0, len(X_tr), BATCH):
            idx = perm[i:i + BATCH]
            element = F.binary_cross_entropy_with_logits(
                head(X_tr[idx]), Y_tr[idx], pos_weight=pw, reduction="none")
            loss = (element * W_tr[idx, None]).sum() / W_tr[idx].sum().clamp(min=1.0) / 12
            opt.zero_grad()
            loss.backward()
            opt.step()

        head.eval()
        with torch.no_grad():
            preds = torch.sigmoid(head(X_ev)).numpy().astype(np.float64)
        auc, _ = macro_auc(Y_ev.astype(np.float64), preds)
        if np.isfinite(auc) and auc > best_auc:
            best_auc, best_pred = auc, preds

    return preds, best_pred, best_auc

## 6. Evaluation protocolThe original notebook validates on the 12 studies left over from a single80/20 split of the 56. That is too small to carry a headline number: twelvestudies, with several of the twelve targets holding one or two positives, so asingle study flipping moves macro AUC by several points. It is the reason thereported 0.6559 is optimistic.So the number here comes from **5-fold cross-validation over all 56 labeledstudies**, with out-of-fold predictions pooled into a single macro AUC. Thereport-labeled pool is disjoint from all 56, so it is reused in every foldwithout leakage.Two metrics, and the distinction matters:- **PRIMARY** -- a fixed 80 epochs. Nothing is chosen using the evaluation  data, so the number is honest.- **SECONDARY** -- best epoch by the held-out fold's own AUC. This peeks at  the evaluation data and is therefore optimistic. It is reported only because  it is the original notebook's protocol, and so is the like-for-like  comparison against 0.6559.

In [ ]:
def cross_validate(n_weak, w_weak, seed, n_splits=5):
    strata = pd.qcut(pd.Series(Y_clean.sum(1)).rank(method="first"),
                     q=n_splits, labels=False)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=20261020)

    oof_final = np.full_like(Y_clean, np.nan, dtype=np.float64)
    oof_best = np.full_like(Y_clean, np.nan, dtype=np.float64)

    for tr, ev in skf.split(np.zeros(len(Y_clean)), strata):
        X_tr = torch.cat([X_clean[tr], X_weak[:n_weak]])
        Y_tr = torch.cat([torch.tensor(Y_clean[tr]), Y_weak[:n_weak]])
        W_tr = torch.cat([torch.ones(len(tr)), torch.full((n_weak,), w_weak)])
        final, best, _ = fit(X_tr, Y_tr, W_tr, X_clean[ev], Y_clean[ev], seed)
        oof_final[ev], oof_best[ev] = final, best

    yt = Y_clean.astype(np.float64)
    primary, per_target = macro_auc(yt, oof_final)
    secondary, _ = macro_auc(yt, oof_best)
    return primary, secondary, per_target


arms = [
    ("image labels only", 0, 1.0),
    ("+ report labels, w=1.0", len(X_weak), 1.0),
    ("+ report labels, w=0.3", len(X_weak), 0.3),
]

results, per_target_store = [], {}
for name, n_weak, w_weak in arms:
    runs = [cross_validate(n_weak, w_weak, s) for s in (0, 1, 2)]
    primary = [r[0] for r in runs]
    secondary = [r[1] for r in runs]
    per_target_store[name] = np.nanmean([r[2] for r in runs], axis=0)
    results.append({
        "arm": name,
        "primary": round(float(np.mean(primary)), 4),
        "primary_sd": round(float(np.std(primary)), 4),
        "secondary": round(float(np.mean(secondary)), 4),
    })
    print(f"{name:26s} PRIMARY {np.mean(primary):.4f} +- {np.std(primary):.4f}"
          f"   SECONDARY {np.mean(secondary):.4f}", flush=True)

print()
print(pd.DataFrame(results).to_string(index=False))

In [ ]:
per_target = pd.DataFrame(per_target_store, index=TARGETS).round(3)
per_target["gain"] = (per_target["+ report labels, w=1.0"]
                      - per_target["image labels only"]).round(3)
print("per-target macro AUC (PRIMARY metric):")
print(per_target.sort_values("gain", ascending=False).to_string())

## 7. Where the gain saturatesWorth running if you plan to label more reports: it answers whether morelabels would still help. Adding report labels in blocks and re-measuringshowed the curve rising to roughly 1,000 labels and then flattening, which iswhy labeling stopped at 1,720 rather than continuing through all 4,094unlabeled studies.Each point is seconds of CPU because the features are cached.

In [ ]:
sweep = []
for n_weak in [0, 200, 400, 800, 1200, len(X_weak)]:
    if n_weak > len(X_weak):
        continue
    aucs = [cross_validate(n_weak, 1.0, s)[0] for s in (0, 1, 2)]
    sweep.append({"n_report_labels": n_weak,
                  "macro_auc": round(float(np.mean(aucs)), 4),
                  "sd": round(float(np.std(aucs)), 4)})
    print(f"n={n_weak:5d}  {np.mean(aucs):.4f} +- {np.std(aucs):.4f}", flush=True)

print()
print(pd.DataFrame(sweep).to_string(index=False))

## 8. Final modelTrained on everything -- all 56 image-labeled studies plus all 1,720report-labeled ones -- at the fixed epoch count whose generalization wasmeasured in section 6. Saved as a full `ConvNeXtFrozenPool` state dict, so itloads into the original notebook's class with `strict=True`.The in-sample AUC printed at the end is not a performance estimate; thesection 6 CV number is.

In [ ]:
class ConvNeXtFrozenPool(nn.Module):
    # unchanged from single-plane-10ep.ipynb

    def __init__(self, num_targets=12, hidden_dim=256, dropout=0.3, slice_batch_size=8):
        super().__init__()
        weights = models.ConvNeXt_Small_Weights.DEFAULT
        self.backbone = models.convnext_small(weights=weights)
        self.feature_dim = self.backbone.classifier[2].in_features
        self.backbone.classifier = nn.Identity()
        self.backbone.requires_grad_(False)
        self.backbone.eval()
        self.slice_batch_size = slice_batch_size
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.feature_dim),
            nn.Linear(self.feature_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_targets),
        )

    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()
        return self

    @torch.no_grad()
    def extract_features(self, images):
        B, S, C, H, W = images.shape
        images = images.reshape(B * S, C, H, W)
        feats = [self.backbone(images[s:s + self.slice_batch_size])
                 for s in range(0, len(images), self.slice_batch_size)]
        return torch.cat(feats, 0).reshape(B, S, -1)

    def forward(self, images, lengths):
        features = self.extract_features(images)
        lengths = lengths.to(features.device).long()
        S = features.shape[1]
        mask = (torch.arange(S, device=features.device)[None, :]
                < lengths[:, None]).unsqueeze(-1).to(features.dtype)
        pooled = (features * mask).sum(1) / mask.sum(1).clamp_min(1.0)
        return self.classifier(pooled)


SEED, W_WEAK = 0, 1.0

X_all = torch.cat([X_clean, X_weak])
Y_all = torch.cat([torch.tensor(Y_clean), Y_weak])
W_all = torch.cat([torch.ones(len(X_clean)), torch.full((len(X_weak),), W_WEAK)])

head = make_head(SEED)
opt = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WD)
pw = compute_pos_weight(Y_all)
gen = torch.Generator().manual_seed(SEED + 1000)

for epoch in range(EPOCHS):
    head.train()
    perm = torch.randperm(len(X_all), generator=gen)
    running = 0.0
    for i in range(0, len(X_all), BATCH):
        idx = perm[i:i + BATCH]
        element = F.binary_cross_entropy_with_logits(
            head(X_all[idx]), Y_all[idx], pos_weight=pw, reduction="none")
        loss = (element * W_all[idx, None]).sum() / W_all[idx].sum().clamp(min=1.0) / 12
        opt.zero_grad()
        loss.backward()
        opt.step()
        running += float(loss.detach()) * len(idx)
    if (epoch + 1) % 20 == 0:
        print(f"epoch {epoch + 1:3d}  train loss {running / len(X_all):.5f}")

full = models.convnext_small(weights=models.ConvNeXt_Small_Weights.DEFAULT)
full.classifier = nn.Identity()
state = {f"backbone.{k}": v for k, v in full.state_dict().items()}
for k, v in head.state_dict().items():
    state[f"classifier.{k}"] = v

cv_row = next(r for r in results if r["arm"] == "+ report labels, w=1.0")
out_path = "/kaggle/working/convnext_frozenpool_weaklabels.pt"
torch.save({
    "epoch": EPOCHS,
    "model_state_dict": state,
    "targets": TARGETS,
    "model_type": "ConvNeXtFrozenPool",
    "cv_macro_auc_56_studies": cv_row["primary"],
    "note": ("head trained on 56 image-labeled studies plus 1,720 studies "
             "labeled from their radiology reports; frozen ImageNet "
             "ConvNeXt-Small backbone; fixed 80 epochs, no epoch selection."),
}, out_path)

reloaded = ConvNeXtFrozenPool()
reloaded.load_state_dict(torch.load(out_path, map_location="cpu")["model_state_dict"], strict=True)
print("saved and strict-load verified:", out_path)

head.eval()
with torch.no_grad():
    in_sample = torch.sigmoid(head(X_clean)).numpy().astype(np.float64)
print(f"in-sample macro AUC on the 56 (NOT a performance estimate): "
      f"{macro_auc(Y_clean.astype(np.float64), in_sample)[0]:.4f}")

## 9. What the numbers do and do not say**The gain is real.** 0.609 to 0.733 PRIMARY, 0.632 to 0.750 SECONDARY, seedspread 0.002-0.009, and it holds at both report-label weights. Per target itappears everywhere except MCL and synovitis, both of which stay near chance.**Why those two fail, and how to predict it.** The positive-rate table insection 3 is the explanation. Image labelers mark synovitis in 45% of the 56studies but the reports support it in only 12%; fracture is 30% against 8%,lateral meniscus 41% against 18%. A report records what the radiologist choseto dictate, not everything an imaging reader would score, so for a findingthat is often present but rarely dictated the report label is mostly zeros andcarries little signal. That is a property of the two modalities disagreeing,not of the rubric, and the comparison in section 3 predicts it before anytraining: where the two positive rates diverge badly, expect no gain. MCL isadditionally confounded because the rubric deliberately excludes MPFL andretinacular injuries, which reports often discuss in the same breath.**The 0.80 figure.** On the 12-study split the original notebook validates on,this pipeline reaches 0.800 +- 0.015, and that is reproducible. It should notbe quoted. Twelve studies with one or two positives in several targets doesnot measure to two decimal places, which is also why 0.6559 was never really0.6559. On the 56-study CV the same pipeline gives 0.733. Nothing here wastuned to reach any particular target.**Two caveats belong with the checkpoint.** The reported figures come from thepublic JPEG-224 re-encode of the images rather than the original DICOMs(near-lossless for a pipeline that quantizes to 8-bit 224x224 anyway, but notidentical). And 1,720 of the training labels were read off reports by a model,not measured from images, so they are wrong in ways image labels are not --a report that omits a finding yields a 0, whether the finding was absent orsimply unmentioned.**Where the remaining headroom is.** Not in more report labels; section 7shows that curve flat past ~1,000. The backbone is the live lever: it isfrozen ImageNet ConvNeXt-Small, and the head sees one mean-pooled vector perstudy, so slice-level information is discarded before the classifier ever seesit. With 1,776 usable training studies instead of 44, unfreezing the backboneor training a real multi-slice aggregator is now affordable -- that is thenext thing worth trying.